# Exercise 2: A U-Net for Wildfire Burn Scar Mapping

This exercise builds a convolutional neural network from scratch and trains it to map wildfire burn scars from satellite imagery. The network architecture used is U-Net, a widely used design for pixel wise segmentation of images. Rather than fine tuning an existing model, you will build every layer yourself, train the network on real data, and inspect how well it learns.

## Dataset

The exercise uses the HLS Burn Scars dataset, published by NASA and IBM. It contains 804 scenes of 512 by 512 pixels, derived from the Harmonized Landsat and Sentinel 2 (HLS) product, covering wildfires across the contiguous United States between 2018 and 2021. Each scene has six spectral bands: Blue, Green, Red, Near Infrared, Shortwave Infrared 1 and Shortwave Infrared 2. Each scene is paired with a mask that labels every pixel as burned, not burned, or missing data.

## What you will learn

1. How to load and normalise multispectral satellite imagery for a deep learning pipeline
2. How to implement the U-Net architecture in PyTorch, layer by layer
3. How to train a segmentation network with a masked loss function that ignores missing data
4. How to evaluate a trained model visually and with Intersection over Union (IoU)

## Setting up the environment

The cell below installs the packages required for this exercise and imports them. PyTorch is used for the model and training loop. Rasterio is used to read the GeoTIFF files that make up the dataset.

In [ ]:
!pip install torch torchvision rasterio matplotlib numpy scikit-learn -q

In [ ]:
import glob
import random

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Downloading the dataset

The dataset is hosted on Hugging Face as a single archive. The cell below downloads and extracts it. The archive is about 2.6 gigabytes, so this step can take a few minutes depending on the connection speed.

Once extracted, the archive produces two folders, `training` and `validation`. Inside each folder, every scene is stored as a file ending in `_merged.tif`, with a matching mask file that has the same name but ends in `.mask.tif` instead.

In [ ]:
!wget -q https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars/resolve/main/hls_burn_scars.tar.gz -O hls_burn_scars.tar.gz
!tar -xzf hls_burn_scars.tar.gz

In [ ]:
train_scene_files = sorted(glob.glob("training/*_merged.tif"))
val_scene_files = sorted(glob.glob("validation/*_merged.tif"))

print(f"Training scenes found   : {len(train_scene_files)}")
print(f"Validation scenes found : {len(val_scene_files)}")

## Looking at a sample scene

Before building a model it is good practice to look at the data. The cell below opens one scene and its mask and displays a false colour composite made from the Shortwave Infrared 1, Near Infrared and Red bands. This band combination is commonly used for fire mapping because burned vegetation and healthy vegetation appear very different from one another in it, unlike in a normal true colour image.

The six bands in each file are stored in the order Blue, Green, Red, Near Infrared, Shortwave Infrared 1, Shortwave Infrared 2, so the indices used below are 4 for Shortwave Infrared 1, 3 for Near Infrared and 2 for Red.

In [ ]:
def load_scene(scene_path):
    """Load a six band scene as a float32 array with shape (6, H, W)."""
    with rasterio.open(scene_path) as src:
        image = src.read().astype(np.float32)
    return image


def load_mask(scene_path):
    """Load the mask that matches a scene. Values are 0 (not burned), 1 (burned) or -1 (missing data)."""
    mask_path = scene_path.replace("_merged.tif", ".mask.tif")
    with rasterio.open(mask_path) as src:
        mask = src.read(1).astype(np.int64)
    return mask


def false_colour_composite(image):
    """Build a Shortwave Infrared, Near Infrared, Red composite for display, scaled to 0 to 1."""
    composite = image[[4, 3, 2], :, :]
    composite = np.moveaxis(composite, 0, -1)
    composite = np.clip(composite / 4000.0, 0, 1)
    return composite


sample_path = train_scene_files[0]
sample_image = load_scene(sample_path)
sample_mask = load_mask(sample_path)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(false_colour_composite(sample_image))
axes[0].set_title("False colour composite (SWIR1, NIR, Red)")
axes[1].imshow(sample_mask, cmap="RdYlGn_r", vmin=-1, vmax=1)
axes[1].set_title("Mask: burned (1), not burned (0), missing (-1)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Image shape: {sample_image.shape}")
print(f"Mask classes present: {np.unique(sample_mask)}")

## Building a PyTorch dataset

PyTorch expects data to be wrapped in a `Dataset` object, which knows how many samples exist and how to load one sample given its index. The dataset class below reads a scene and its mask from disk, scales the reflectance values to approximately the 0 to 1 range, and optionally applies a simple augmentation.

HLS reflectance values are stored as integers scaled by 10000. Dividing by this factor and clipping to the 0 to 1 range gives a reasonable normalisation without needing to compute dataset wide statistics.

In [ ]:
def random_flip_and_rotate(image, mask):
    """Apply the same random flip and 90 degree rotation to an image and its mask."""
    if random.random() < 0.5:
        image = torch.flip(image, dims=[2])
        mask = torch.flip(mask, dims=[1])
    if random.random() < 0.5:
        image = torch.flip(image, dims=[1])
        mask = torch.flip(mask, dims=[0])
    number_of_rotations = random.randint(0, 3)
    if number_of_rotations > 0:
        image = torch.rot90(image, number_of_rotations, dims=[1, 2])
        mask = torch.rot90(mask, number_of_rotations, dims=[0, 1])
    return image, mask


class BurnScarDataset(Dataset):
    def __init__(self, scene_files, augment=False):
        self.scene_files = scene_files
        self.augment = augment

    def __len__(self):
        return len(self.scene_files)

    def __getitem__(self, index):
        scene_path = self.scene_files[index]
        image = load_scene(scene_path)
        mask = load_mask(scene_path)

        image = np.clip(image / 10000.0, 0.0, 1.0)

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)

        if self.augment:
            image, mask = random_flip_and_rotate(image, mask)

        return image, mask


train_dataset = BurnScarDataset(train_scene_files, augment=True)
val_dataset = BurnScarDataset(val_scene_files, augment=False)

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

sample_image_batch, sample_mask_batch = next(iter(train_loader))
print(f"Image batch shape: {sample_image_batch.shape}")
print(f"Mask batch shape : {sample_mask_batch.shape}")

## Defining the U-Net architecture

U-Net has two halves. The encoder repeatedly applies convolutions and downsamples the image, building up an understanding of increasingly large scale patterns while losing spatial detail. The decoder does the reverse, upsampling step by step back to the original resolution. At each decoder stage, the corresponding encoder features are concatenated back in through a skip connection, so that fine spatial detail lost during downsampling can be recovered.

We first define a small helper function that creates a block of two convolutional layers with a ReLU activation, since this pattern is repeated many times throughout the network. The full network is then assembled from these blocks.

In [ ]:
def conv_block(in_channels, out_channels):
    """Two 3 by 3 convolutions, each followed by a ReLU activation."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
    )

class UNet(nn.Module):
    def __init__(self, in_channels=6, num_classes=2):
        super().__init__()

        # Encoder
        self.enc1 = conv_block(in_channels, 32)
        self.enc2 = conv_block(32, 64)
        self.enc3 = conv_block(64, 128)
        self.enc4 = conv_block(128, 256)
        self.pool = nn.MaxPool2d(kernel_size=2)

        # Bottleneck
        self.bottleneck = conv_block(256, 512)

        # Decoder, each stage upsamples then concatenates the matching encoder features
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = conv_block(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = conv_block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = conv_block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = conv_block(64, 32)

        self.classifier = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.classifier(d1)


model = UNet(in_channels=6, num_classes=2).to(device)
total_parameters = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_parameters:,}")

## Loss function and the missing data class

The mask contains three values: 0 for not burned, 1 for burned, and -1 for missing data. Missing data pixels should not influence training at all. PyTorch's cross entropy loss supports this directly through its `ignore_index` argument, which simply excludes any pixel with that label from the loss calculation, without requiring any manual masking code.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Training and validation loop

Training a segmentation network means repeating the same steps for every batch of images: run the batch through the model, compare the prediction with the ground truth mask using the loss function, and update the model weights to reduce that loss. The same forward pass is also used during validation, except that no weight update happens, which lets us check whether the model is learning to generalise rather than only memorising the training scenes.

The function below runs one full pass over a data loader, either in training mode or in evaluation mode, and returns the average loss and the pixel accuracy on the pixels that are not missing data.

In [ ]:
def run_one_epoch(model, loader, optimizer, criterion, device, training):
    model.train() if training else model.eval()

    total_loss = 0.0
    correct_pixels = 0
    valid_pixels = 0

    torch.set_grad_enabled(training)
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        if training:
            optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, masks)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = torch.argmax(logits, dim=1)
        valid = masks != -1
        correct_pixels += ((predictions == masks) & valid).sum().item()
        valid_pixels += valid.sum().item()

    average_loss = total_loss / len(loader.dataset)
    pixel_accuracy = correct_pixels / valid_pixels
    return average_loss, pixel_accuracy

In [ ]:
NUM_EPOCHS = 15

history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_accuracy = run_one_epoch(model, train_loader, optimizer, criterion, device, training=True)
    val_loss, val_accuracy = run_one_epoch(model, val_loader, optimizer, criterion, device, training=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)

    print(f"Epoch {epoch + 1:2d}/{NUM_EPOCHS}  "
          f"train loss {train_loss:.4f}  val loss {val_loss:.4f}  "
          f"train accuracy {train_accuracy:.4f}  val accuracy {val_accuracy:.4f}")

## Inspecting the learning curves

Plotting the training and validation loss and accuracy over the epochs is the simplest way to judge whether training went well. If the validation loss stops improving while the training loss keeps decreasing, the model is beginning to overfit to the training scenes. Exercise 5 covers this diagnosis in more depth.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history["train_loss"], label="Training loss")
axes[0].plot(history["val_loss"], label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross entropy loss")
axes[0].set_title("Loss over training")
axes[0].legend()

axes[1].plot(history["train_accuracy"], label="Training accuracy")
axes[1].plot(history["val_accuracy"], label="Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Pixel accuracy")
axes[1].set_title("Accuracy over training")
axes[1].legend()

plt.tight_layout()
plt.show()

## Evaluating with Intersection over Union

Pixel accuracy can be misleading for this task, since most pixels in a scene are not burned and a model could reach high accuracy by always predicting the majority class. Intersection over Union (IoU) is a more informative metric for segmentation, since it measures how well the predicted burned area overlaps with the true burned area, independently of how common that class is. The function below computes IoU separately for each class, while excluding missing data pixels from the calculation.

In [ ]:
def per_class_iou(model, loader, device, num_classes=2):
    model.eval()
    intersection = np.zeros(num_classes)
    union = np.zeros(num_classes)

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            predictions = torch.argmax(model(images), dim=1)

            valid = masks != -1
            for class_id in range(num_classes):
                predicted_class = (predictions == class_id) & valid
                true_class = (masks == class_id) & valid
                intersection[class_id] += (predicted_class & true_class).sum().item()
                union[class_id] += (predicted_class | true_class).sum().item()

    return intersection / np.clip(union, 1, None)


class_names = ["Not burned", "Burned"]
iou_scores = per_class_iou(model, val_loader, device)
for name, score in zip(class_names, iou_scores):
    print(f"{name:12s} IoU: {score:.4f}")
print(f"Mean IoU: {iou_scores.mean():.4f}")

## Visualising predictions

Numbers alone rarely tell the full story of how a segmentation model behaves. The cell below picks a few validation scenes at random and displays the false colour composite, the ground truth mask and the predicted mask side by side, so the errors the model makes can be inspected directly.

In [ ]:
def plot_predictions(model, dataset, device, num_samples=4, seed=7):
    model.eval()
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(dataset), num_samples, replace=False)

    fig, axes = plt.subplots(num_samples, 3, figsize=(11, 3.6 * num_samples))
    column_titles = ["False colour composite", "Ground truth", "Prediction"]
    for column, title in enumerate(column_titles):
        axes[0, column].set_title(title)

    with torch.no_grad():
        for row, index in enumerate(indices):
            image, mask = dataset[index]
            logits = model(image.unsqueeze(0).to(device))
            prediction = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

            axes[row, 0].imshow(false_colour_composite(image.numpy()))
            axes[row, 1].imshow(mask.numpy(), cmap="RdYlGn_r", vmin=-1, vmax=1)
            axes[row, 2].imshow(prediction, cmap="RdYlGn_r", vmin=-1, vmax=1)

            for ax in axes[row]:
                ax.axis("off")

    legend_handles = [
        mpatches.Patch(color="green", label="Not burned"),
        mpatches.Patch(color="red", label="Burned"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=2)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


plot_predictions(model, val_dataset, device)

## Summary

This exercise built a complete segmentation pipeline without relying on any pre trained model. The main steps were reading multiband GeoTIFF imagery into a PyTorch dataset, normalising reflectance values, building the U-Net architecture explicitly from convolutional blocks, and training it with a loss function that ignores missing data pixels.

The next exercise contrasts this approach with a Vision Transformer that starts from pre trained weights, applied to a different mapping problem, so that the two architectures and the two training strategies (training from scratch versus fine tuning) can be compared directly.